# Can we predict `passed_quiz`?
Target: whether a student passes the quiz closing a tutoring session (1 = pass). Rows are **sessions**; the same student appears in many rows.

Plan: inspect the data, pick a validation scheme that matches the real use case (predicting for students the model hasn't seen), compare simple models to a base-rate baseline, then check what drives the signal.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss, accuracy_score
from sklearn.inspection import permutation_importance
from sklearn.calibration import calibration_curve
pd.set_option('display.width', 140)
SEED = 0
df = pd.read_csv('data.csv')
print(pd.read_csv('columns.csv').to_string(index=False))
df.shape

## 1. Data checks

In [ ]:
print('missing:', df.isna().sum().sum(), '| duplicate rows:', df.duplicated().sum())
g = df.groupby('student_key')
print('students:', df.student_key.nunique(), '| sessions/student (min, median, max):', g.size().describe()[['min', '50%', 'max']].tolist())
print('pass rate:', round(df.passed_quiz.mean(), 3))
print('students with >1 school / enrol_year:', (g.school_code.nunique() > 1).sum(), (g.enrol_year.nunique() > 1).sum())
print('students whose prior_score differs between sessions:', (g.prior_score.nunique() > 1).sum(), 'of', len(g))
print('within-student SD of prior_score (median):', round(g.prior_score.std().median(), 2), 'vs overall SD', round(df.prior_score.std(), 2))
df.describe().T

Findings:
- No missing values or duplicates; classes are mildly imbalanced (~42% pass).
- **Rows are not independent**: ~7 sessions per student. `school_code` and `enrol_year` are constant per student. `prior_score` is described as the placement score "as stored on the session record", yet it jitters slightly between a student's sessions (median within-student SD about 0.3 vs about 12 overall), so it is effectively a student-level constant plus noise. That makes it close to a student identifier, so a random row split would leak student identity. **All validation below is grouped by student.**

## 2. Univariate relationships

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(17, 3.4))
for a, c, q in zip(ax, ['prior_score', 'session_minutes', 'hints_used', 'enrol_year'], [10, 10, None, None]):
    x = pd.qcut(df[c], q, duplicates='drop') if q else df[c]
    t = df.groupby(x, observed=True).passed_quiz.agg(['mean', 'size'])
    t = t[t['size'] >= 30]
    a.plot(range(len(t)), t['mean'], 'o-')
    a.set_xticks(range(len(t)))
    a.set_xticklabels([str(round(i.mid)) if q else i for i in t.index], rotation=45, fontsize=7)
    a.axhline(df.passed_quiz.mean(), ls=':', c='gray'); a.set_title(c); a.set_ylim(0.2, 0.65)
ax[0].set_ylabel('pass rate'); plt.tight_layout(); plt.show()
sp = df.groupby('student_key').passed_quiz.agg(['mean', 'size']); sp = sp[sp['size'] >= 5]
pr = df.passed_quiz.mean()
print('SD of student-level pass rate (students with >=5 sessions):', round(sp['mean'].std(), 3))
print('expected SD if every session were an independent coin flip at the base rate:', round(np.sqrt(pr * (1 - pr) / sp['size']).mean(), 3))

- More time in session means a higher pass rate; more hints means lower. `prior_score`, `enrol_year` and `school_code` look nearly flat.
- Per-student pass rates are far more spread out than chance would give, so there is a **student-level effect that the columns don't explain**. Student identity carries information, but it is unavailable for new students.

## 3. Model comparison with grouped cross-validation
Features: `prior_score, session_minutes, hints_used, enrol_year, school_code`. `student_key` is **not** a feature. Folds keep each student wholly in train or test.

In [ ]:
feats = ['prior_score', 'session_minutes', 'hints_used', 'enrol_year', 'school_code']
X, y, grp = df[feats], df.passed_quiz.values, df.student_key.values
lr = lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
gbm = lambda: HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=SEED)
specs = {
    'base rate': (DummyClassifier(strategy='prior'), feats),
    'logistic (minutes + hints)': (lr(), ['session_minutes', 'hints_used']),
    'logistic (all)': (lr(), feats),
    'grad. boosting (all)': (gbm(), feats),
}
rows, oofs = [], {}
for name, (m, cols) in specs.items():
    cv = StratifiedGroupKFold(5, shuffle=True, random_state=SEED)
    oof = np.zeros(len(df)); aucs = []
    for tr, te in cv.split(X, y, grp):
        m.fit(X.iloc[tr][cols], y[tr]); p = m.predict_proba(X.iloc[te][cols])[:, 1]
        oof[te] = p; aucs.append(roc_auc_score(y[te], p))
    oofs[name] = oof
    rows.append(dict(model=name, AUC=np.mean(aucs), AUC_sd_over_folds=np.std(aucs), logloss=log_loss(y, oof),
                     brier=brier_score_loss(y, oof), acc_at_0_5=accuracy_score(y, oof > .5)))
pd.DataFrame(rows).set_index('model').round(4)

AUC 0.5 means no skill. Accuracy should be compared with the majority-class rate, not with 50%.

In [ ]:
print('majority-class accuracy:', round(1 - y.mean(), 3))
m = lr().fit(X, y)
print(pd.Series(m[-1].coef_[0], index=feats).round(3).rename('standardised logistic coef (all data, descriptive)'))
tr, te = next(StratifiedGroupKFold(5, shuffle=True, random_state=SEED).split(X, y, grp))
g_ = gbm().fit(X.iloc[tr], y[tr])
pi = permutation_importance(g_, X.iloc[te], y[te], scoring='roc_auc', n_repeats=20, random_state=SEED)
print(pd.Series(pi.importances_mean, index=feats).round(4).rename('AUC drop when shuffled (held-out students)'))

## 4. Calibration and score separation

In [ ]:
p = oofs['logistic (all)']
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
fp, mp = calibration_curve(y, p, n_bins=10, strategy='quantile')
ax[0].plot(mp, fp, 'o-'); ax[0].plot([.2, .7], [.2, .7], ':', c='gray')
ax[0].set_xlabel('predicted'); ax[0].set_ylabel('observed pass rate'); ax[0].set_title('Calibration (out-of-fold)')
ax[1].hist(p[y == 0], bins=30, alpha=.6, label='failed'); ax[1].hist(p[y == 1], bins=30, alpha=.6, label='passed')
ax[1].legend(); ax[1].set_title('Predicted probability')
plt.tight_layout(); plt.show()
dec = pd.qcut(p, 10, labels=False)
print(pd.Series(y).groupby(dec).mean().round(3).rename('observed pass rate by predicted decile').to_string())

## 5. Secondary question: does the student's other-session history help?
For a *returning* student, the pass rate on their other sessions is a candidate feature. To avoid leaking the row's own label, each test row gets the student's pass rate computed **only from sessions in the training folds** (shrunk toward the base rate; students with no training sessions get the base rate). Folds are random by row, since here students are meant to be shared.

Caveat: the data has **no session timestamp or order**, so the "history" may include sessions that happen after the one being predicted. Read this as an upper bound on what history could give, not a deployable estimate.

(A simpler leave-one-out version of this feature gave AUC about 0.85 in a first attempt, but that was an artefact: `prior_score` is nearly unique per student, so a flexible model can decode the LOO quirk and recover the label. It is not used here.)

In [ ]:
def hist_rate(tr_idx, idx, k=5):
    st = df.iloc[tr_idx].groupby('student_key').passed_quiz.agg(['sum', 'size'])
    j = df.iloc[idx][['student_key']].join(st, on='student_key').fillna(0)
    return ((j['sum'] + k * pr) / (j['size'] + k)).values

oof_h, oof_b = np.zeros(len(df)), np.zeros(len(df))
for tr, te in StratifiedKFold(5, shuffle=True, random_state=SEED).split(df, y):
    # training-row feature must also be out-of-sample: build it with an inner split
    rate_tr = np.zeros(len(tr))
    for itr, ite in StratifiedKFold(5, shuffle=True, random_state=SEED).split(tr, y[tr]):
        rate_tr[ite] = hist_rate(tr[itr], tr[ite])
    Xtr = df.iloc[tr][feats].assign(hist=rate_tr); Xte = df.iloc[te][feats].assign(hist=hist_rate(tr, te))
    oof_h[te] = lr().fit(Xtr, y[tr]).predict_proba(Xte)[:, 1]
    oof_b[te] = lr().fit(Xtr[feats], y[tr]).predict_proba(Xte[feats])[:, 1]
print('random-row folds, logistic without history: AUC', round(roc_auc_score(y, oof_b), 4), '| log loss', round(log_loss(y, oof_b), 4))
print('random-row folds, logistic with student history: AUC', round(roc_auc_score(y, oof_h), 4), '| log loss', round(log_loss(y, oof_h), 4))

## Conclusions
- **For a new student, prediction is weak.** Grouped CV AUC is about 0.60 (0.5 = chance), accuracy about 0.60 against 0.58 for always predicting "fail". The model separates risk deciles (about 28% to 57% pass), so it ranks sessions better than chance but cannot reliably call individual outcomes.
- **What carries the signal:** `session_minutes` (longer, more passes) and `hints_used` (more hints, fewer passes). `prior_score`, `enrol_year` and `school_code` add essentially nothing. Gradient boosting does no better than logistic regression (it overfits slightly), so a simple model is enough.
- **Much stronger: knowing the student.** Pass rates differ between students far more than chance allows, and a student's other sessions lift AUC to about 0.84. That only applies to returning students, and the data has no timestamps, so we can't tell whether this history would have been available *before* the session. Confirm that before relying on it.
- Not checked: whether `session_minutes` and `hints_used` are known before the quiz (both are session-level and could be recorded at the end, which is fine if predictions are made after the session but not if made at its start), and any time-based drift (only `enrol_year`, which is flat).